# Phase 2 调试 Notebook — Tri-Plane 三平面管线逐项检查

> 在 Kaggle 环境或有真实 DICOM 数据的机器上逐 cell 运行，快速定位 tri-plane 管线问题。
>
> 如果没有 DICOM 数据 (本地环境)，元数据分析、模型验证、融合模块仍可完整测试。

## 检查清单
| # | 检查项 | 数据依赖 | 预期 |
|---|--------|----------|------|
| 1 | 环境 & 导入 | 无 | 无报错 |
| 2 | 配置加载 | 无 | YAML 解析成功，关键 key 存在 |
| 3 | 伪标签加载 & 合并 | pseudo_labels.csv | train/val split 合理 |
| 4 | Series 元数据分析 | train_series.csv | 三平面覆盖率统计 |
| 5 | TriPlaneDataset 构建 | DICOM | 样本数 > 0, 三平面 key 存在 |
| 6 | 三平面可视化 | DICOM | 3×5 切片 montage |
| 7 | 模型 forward pass | 无 | [B,5,H,W]×3 → [B,12] |
| 8 | 缺失平面处理 | 无 | 全零输入不崩溃，missing_emb 生效 |
| 9 | 融合模块对比 | 无 | concat vs transformer dim |
| 10 | 过拟合测试 | DICOM | 单 batch loss→0 |
| 11 | 快速干跑 (2 epochs) | DICOM | 不报错，梯度累积正确 |
| 12 | 梯度累积验证 | DICOM | optimizer.step() 频率 = batch / accum |
| 13 | 平面特征分析 | DICOM | 各平面贡献度对比 |
| 14 | 推理管线测试 | 模型 ckpt + test DICOM | 生成 submission CSV |

In [ ]:
# ============================================================
# Cell 1: 环境 & 导入
# ============================================================
from __future__ import annotations
import sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt

deps = {}
for name in ["torch", "numpy", "pandas", "yaml", "cv2", "pydicom", "timm", "sklearn"]:
    try:
        __import__(name)
        deps[name] = "OK"
    except ImportError:
        deps[name] = "MISSING"

print(f"Python {sys.version.split()[0]}  |  PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB")
print("  " + "  ".join(f"{k}:{v}" for k, v in deps.items()))

In [ ]:
# ============================================================
# Cell 2: 配置加载 & 检查
# ============================================================
CONFIG_PATH = PROJECT_ROOT / "configs" / "triplane.yaml"
with open(CONFIG_PATH, encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 必要 key 检查
for section in ["experiment", "paths", "data", "model", "train", "loss", "pseudo_label"]:
    ok = "OK" if section in config else "MISSING"
    print(f"  [{ok}] {section}")

print(f"\n  Stage:       {config['experiment']['stage']}")
print(f"  Model arch:  {config['model']['arch']}")
print(f"  Fusion:      {config['model']['fusion']}")
print(f"  Shared bb:   {config['model']['shared_backbone']}")
print(f"  Planes:      {config['data']['planes']}")
print(f"  Batch size:  {config['train']['batch_size']} × {config['train']['gradient_accumulation_steps']} accum")
print(f"  Pseudo conf: {config['pseudo_label']['confidence']}")
print(f"  LR:          {config['optimizer']['lr']}")
print(f"  Focal γ/α:   {config['loss']['gamma']} / {config['loss']['alpha']}")

# 路径存在性
print(f"\n  路径检查:")
for k, v in config["paths"].items():
    p = Path(v)
    icon = "EXISTS" if p.exists() else "NOFILE"
    print(f"    [{icon}] {k}: {v}")

In [ ]:
# ============================================================
# Cell 3: 伪标签加载 & 与 Gold Label 合并
# ============================================================
from datasets import PseudoLabelLoader
from utils import TARGET_COLUMNS

pseudo_cfg = config["pseudo_label"]

# 加载伪标签
loader = PseudoLabelLoader(
    pseudo_csv=pseudo_cfg["pseudo_csv"],
    pseudo_valid_csv="data/pseudo_labels_valid.csv",
)

# 统计
stats = loader.stats()
print("伪标签总体统计:")
print(f"  Total:          {stats['total']:,}")
by_conf = stats["by_confidence_overall"]
for level in ["HIGH", "HIGH_PLUS_MEDIUM", "MEDIUM"]:
    if level in by_conf:
        pct = by_conf[level] / stats["total"] * 100
        print(f"  {level:<20s} {by_conf[level]:,} ({pct:.1f}%)")

print(f"\nNLP 伪标签类别分布 (全部 4349 studies):")
balance = stats["class_balance"]
for k, v in sorted(balance.items(), key=lambda x: -x[1]):
    bar = "█" * int(v / max(balance.values()) * 40)
    print(f"  {k:<20s} {v:5d}  {bar}")

# 合并 gold + pseudo
print(f"\n--- 合并 Gold + Pseudo (confidence={pseudo_cfg['confidence']}) ---")
train_df, val_df = loader.merge_with_gold(
    gold_csv=pseudo_cfg["gold_csv"],
    confidence=pseudo_cfg["confidence"],
    val_from_gold=True,
)
print(f"  Train (pseudo): {len(train_df)} studies")
print(f"  Valid (gold):   {len(val_df)} studies")

# 打印 train/val 的类别分布对比
print(f"\n  {'Class':<20s} {'Train+':>8s} {'Val+':>6s} {'Val Rate':>10s}")
print(f"  {'-'*46}")
for col in TARGET_COLUMNS:
    t_pos = int(train_df[col].sum()) if col in train_df.columns else 0
    v_pos = int(val_df[col].sum()) if col in val_df.columns else 0
    v_rate = v_pos / len(val_df) * 100 if len(val_df) > 0 else 0
    print(f"  {col:<20s} {t_pos:8d} {v_pos:6d} {v_rate:9.1f}%")

# NLP 验证报告
report = loader.load_report()
if len(report) > 0:
    print(f"\n--- NLP 验证报告 (在 58 gold 上) ---")
    print(f"  {'Class':<20s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'Acc':>8s}")
    print(f"  {'-'*56}")
    for _, r in report.iterrows():
        print(f"  {r['class']:<20s} {r['precision']:10.3f} {r['recall']:8.3f} {r['f1']:8.3f} {r['accuracy']:8.3f}")

In [ ]:
# ============================================================
# Cell 3b: 伪标签置信度详细分析
# ============================================================
pseudo_raw = loader.load()

# 每个类别的 HIGH/MEDIUM/LOW 分布
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(TARGET_COLUMNS):
    ax = axes[i]
    conf_col = f"conf_{col}"
    if conf_col in pseudo_raw.columns:
        counts = pseudo_raw[conf_col].value_counts()
        colors = {"HIGH": "#2ecc71", "MEDIUM": "#f39c12", "LOW": "#e74c3c", "REVIEW": "#9b59b6", "FAIL": "#95a5a6"}
        bar_colors = [colors.get(k, "#95a5a6") for k in counts.index]
        ax.bar(counts.index, counts.values, color=bar_colors)
        ax.set_title(col, fontsize=10)
        ax.tick_params(labelsize=8)

axes[11].axis("off")
fig.suptitle("Per-Class Confidence Distribution (NLP Pseudo Labels)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# 预测正样本率 vs 置信度
print(f"\n  预测正样本率对比:")
print(f"  {'Class':<20s} {'ALL':>8s} {'HIGH':>8s} {'MEDIUM':>8s}")
print(f"  {'-'*46}")
for col in TARGET_COLUMNS:
    pred_col = f"pred_{col}"
    conf_col = f"conf_{col}"
    if pred_col in pseudo_raw.columns:
        all_rate = pseudo_raw[pred_col].mean() * 100
        high_mask = pseudo_raw[conf_col] == "HIGH"
        med_mask = pseudo_raw[conf_col] == "MEDIUM"
        high_rate = pseudo_raw.loc[high_mask, pred_col].mean() * 100 if high_mask.any() else 0
        med_rate = pseudo_raw.loc[med_mask, pred_col].mean() * 100 if med_mask.any() else 0
        print(f"  {col:<20s} {all_rate:7.1f}% {high_rate:7.1f}% {med_rate:7.1f}%")

In [ ]:
# ============================================================
# Cell 4: Series 元数据分析 — 三平面覆盖率
# ============================================================
# 关键分析: 每个 study 有哪些平面可用？缺失率是多少？
series_df = pd.read_csv(config["paths"]["series_csv"])
print(f"Series 总数: {len(series_df):,}")
print(f"Study 总数:  {series_df['StudyInstanceUID'].nunique():,}")

# 每 study 的平面覆盖
study_planes = series_df.groupby("StudyInstanceUID")["Anatomical_Plane"].apply(set)
plane_counts = series_df["Anatomical_Plane"].value_counts()
print(f"\n  序列级平面分布:")
for p in ["Sagittal", "Coronal", "Axial"]:
    print(f"    {p}: {plane_counts.get(p, 0):,} series")

# 三平面齐全的 study 比例
all_three = study_planes.apply(lambda s: {"Sagittal", "Coronal", "Axial"}.issubset(s))
sag_only = study_planes.apply(lambda s: s == {"Sagittal"})
missing_cor = study_planes.apply(lambda s: "Coronal" not in s)
missing_ax = study_planes.apply(lambda s: "Axial" not in s)

print(f"\n  三平面覆盖率 (Study 级):")
print(f"    三平面齐全: {all_three.sum():,} / {len(study_planes):,} ({all_three.mean()*100:.1f}%)")
print(f"    仅 Sagittal: {sag_only.sum():,} ({sag_only.mean()*100:.1f}%)")
print(f"    缺 Coronal:  {missing_cor.sum():,} ({missing_cor.mean()*100:.1f}%)")
print(f"    缺 Axial:    {missing_ax.sum():,} ({missing_ax.mean()*100:.1f}%)")

# Fluid_Sensitive + Fat_Suppression 分析
if "Fluid_Sensitive" in series_df.columns:
    print(f"\n  T2/PD 序列 (Fluid_Sensitive=1): {(series_df['Fluid_Sensitive']==1).sum():,} series")
if "Fat_Suppression" in series_df.columns:
    print(f"  压脂序列 (Fat_Suppression=1): {(series_df['Fat_Suppression']==1).sum():,} series")
    
# 每 study 的 series 数量分布
series_per_study = series_df.groupby("StudyInstanceUID").size()
print(f"\n  每 Study Series 数分布:")
for i in range(int(series_per_study.min()), min(int(series_per_study.max()) + 1, 12)):
    cnt = (series_per_study == i).sum()
    if cnt > 0:
        print(f"    {i:2d} series: {cnt:5d} studies")

In [ ]:
# ============================================================
# Cell 4b: 与伪标签的交集分析
# ============================================================
# 检查: pseudo-labeled studies 中有多少有三平面 DICOM?

# 交集统计
pseudo_uids = set(train_df.index)
gold_uids = set(val_df.index)
dicom_uids = set(series_df["StudyInstanceUID"].unique())

print(f"  伪标签训练集 studies:     {len(pseudo_uids):,}")
print(f"  Gold 验证集 studies:      {len(gold_uids):,}")
print(f"  DICOM 可用 studies:       {len(dicom_uids):,}")

train_with_dicom = pseudo_uids & dicom_uids
val_with_dicom = gold_uids & dicom_uids
print(f"\n  训练集有 DICOM:           {len(train_with_dicom):,} / {len(pseudo_uids):,} ({len(train_with_dicom)/max(len(pseudo_uids),1)*100:.1f}%)")
print(f"  验证集有 DICOM:           {len(val_with_dicom):,} / {len(gold_uids):,} ({len(val_with_dicom)/max(len(gold_uids),1)*100:.1f}%)")

# 三平面齐全的 train/val
train_all3 = pseudo_uids & set(study_planes[all_three].index)
val_all3 = gold_uids & set(study_planes[all_three].index)
print(f"\n  训练集三平面齐全:         {len(train_all3):,} / {len(pseudo_uids):,}")
print(f"  验证集三平面齐全:         {len(val_all3):,} / {len(gold_uids):,}")

# 如果有大量 study 缺平面，TriPlaneDataset 会用 missing_emb 处理
if len(pseudo_uids - dicom_uids) > 0:
    print(f"\n  ⚠️  {len(pseudo_uids - dicom_uids)} 个训练 study 完全没有 DICOM (将跳过)")
if len(gold_uids - dicom_uids) > 0:
    print(f"  ⚠️  {len(gold_uids - dicom_uids)} 个验证 study 完全没有 DICOM (将跳过)")

In [ ]:
# ============================================================
# Cell 5: TriPlaneDataset 构建
# ============================================================
# 注意: 此 cell 仅在 DICOM 文件存在时产生样本
# 本地无 DICOM 时，可观察跳过统计和空 dataset 行为

from datasets import TriPlaneDataset

ds = TriPlaneDataset(
    series_df=series_df,
    labels_df=train_df,                     # 伪标签训练集
    dicom_root=config["paths"]["dicom_root"],
    image_size=config["data"]["image_size"],
    slice_count=config["data"]["slice_count"],
    planes=config["data"]["planes"],
    is_train=True,
)

print(f"TriPlaneDataset 训练样本数: {len(ds):,}")

# 打印 study_plane_map 的覆盖信息 (前 5 个 study)
for i, (sid, pinfo) in enumerate(ds.study_plane_map.items()):
    if i >= 5:
        break
    planes_ok = []
    for p in ["Sagittal", "Coronal", "Axial"]:
        if p in pinfo and pinfo[p]["exists"]:
            planes_ok.append(f"{p}({pinfo[p]['n_slices']}sl)")
        else:
            planes_ok.append(f"{p}(MISSING)")
    print(f"  {sid[:40]}...  →  {', '.join(planes_ok)}")

In [ ]:
# ============================================================
# Cell 5b: 样本内容检查
# ============================================================
if len(ds) > 0:
    # 取 3 个不同位置的 sample
    indices = [0, len(ds) // 2, len(ds) - 1]
    
    for idx in indices:
        sample = ds[idx]
        print(f"\n  Sample [{idx}]  study: {sample['study_uid'][:50]}...")
        for plane_key in ["sag", "cor", "ax"]:
            t = sample[plane_key]
            has_sig = t.abs().sum() > 0
            sig_str = "signal" if has_sig else "ZEROS (missing)"
            print(f"    {plane_key}: shape={list(t.shape)}  range=[{t.min():.3f}, {t.max():.3f}]  {sig_str}")
        
        active_labels = []
        for i, col in enumerate(TARGET_COLUMNS):
            if sample["labels"][i] > 0:
                active_labels.append(col)
        print(f"    labels: {active_labels or 'all negative'}")
        print(f"    series_uids: sag={sample['series_uids']['sag'][:30]}..., cor={sample['series_uids']['cor'][:30]}..., ax={sample['series_uids']['ax'][:30]}...")
else:
    print("⚠️  Dataset 为空 — DICOM 数据不可用。在 Kaggle 上运行时将产生样本。")
    print("   TriPlaneDataset 已正确初始化，代码结构验证通过。")

In [ ]:
# ============================================================
# Cell 6: 三平面 5-slice 堆叠可视化
# ============================================================
# 仅当 DICOM 数据存在时有效

if len(ds) > 0:
    sample = ds[len(ds) // 3]  # 取中间某个 sample
    
    fig, axes = plt.subplots(3, 5, figsize=(16, 10))
    plane_names = ["Sagittal", "Coronal", "Axial"]
    plane_keys = ["sag", "cor", "ax"]
    
    for row, (pname, pkey) in enumerate(zip(plane_names, plane_keys)):
        stack = sample[pkey]  # [5, H, W]
        for col in range(5):
            ax = axes[row, col]
            ax.imshow(stack[col], cmap="gray")
            title = f"{pname} slice {col-2:+d}" if row == 0 else f"slice {col-2:+d}"
            ax.set_title(title, fontsize=9)
            ax.axis("off")
        
        # 每行左边标平面名
        axes[row, 0].set_ylabel(pname, fontsize=12, rotation=90, labelpad=15)
    
    fig.suptitle(f"Tri-Plane 5-Slice Stacks — Study: {sample['study_uid'][:40]}...", fontsize=13)
    plt.tight_layout()
    plt.show()
    
    # 检查各平面是否存在信号缺失
    for pkey in plane_keys:
        t = sample[pkey]
        if t.abs().sum() == 0:
            print(f"  ⚠️  {pkey}: 全零 (该平面 DICOM 缺失)")
else:
    print("⚠️  无 DICOM 数据，跳过可视化。在 Kaggle 上运行时将显示 3×5 切片 montage。")

In [ ]:
# ============================================================
# Cell 7: 模型 Forward Pass 验证
# ============================================================
from models import TriPlaneModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model_cfg = config["model"]

# 测试 1: 共享 backbone (默认)
print("\n--- 测试 1: 共享 Backbone ---")
model_shared = TriPlaneModel(
    in_channels=model_cfg["in_channels"],
    num_classes=model_cfg["num_classes"],
    feature_dim=model_cfg["feature_dim"],
    pretrained=model_cfg["pretrained"],
    dropout=model_cfg["dropout"],
    shared_backbone=True,
    fusion=model_cfg["fusion"],
).to(device)

n_p = sum(p.numel() for p in model_shared.parameters()) / 1e6
n_tr = sum(p.numel() for p in model_shared.parameters() if p.requires_grad) / 1e6
print(f"  Params: {n_p:.1f}M total, {n_tr:.1f}M trainable")

# Forward test — random input
for bs in [1, 2, 4]:
    sag = torch.randn(bs, 5, 384, 384).to(device)
    cor = torch.randn(bs, 5, 384, 384).to(device)
    ax = torch.randn(bs, 5, 384, 384).to(device)
    with torch.no_grad():
        out = model_shared(sag, cor, ax)
    print(f"  batch={bs}: input {list(sag.shape)} × 3 → output {list(out.shape)}")
    assert out.shape == (bs, 12), f"Shape mismatch!"

print("  OK: Shared backbone shapes correct")

# 测试 2: 独立 backbone
print("\n--- 测试 2: 独立 Backbone (每平面独立权重) ---")
model_indep = TriPlaneModel(
    in_channels=model_cfg["in_channels"],
    num_classes=model_cfg["num_classes"],
    pretrained=False,                         # 不加载预训练加速测试
    shared_backbone=False,
    fusion=model_cfg["fusion"],
).to(device)

n_p_i = sum(p.numel() for p in model_indep.parameters()) / 1e6
print(f"  Params: {n_p_i:.1f}M total (vs {n_p:.1f}M shared)")

sag = torch.randn(2, 5, 384, 384).to(device)
cor = torch.randn(2, 5, 384, 384).to(device)
ax = torch.randn(2, 5, 384, 384).to(device)
with torch.no_grad():
    out_i = model_indep(sag, cor, ax)
print(f"  Forward: {list(out_i.shape)}  OK")

# 对比两种 backbone 的输出差异
print(f"\n  共享 vs 独立 backbone 在同一输入下的差异:")
print(f"    Shared:   logits range [{out.min():.3f}, {out.max():.3f}]")
print(f"    Indep:    logits range [{out_i.min():.3f}, {out_i.max():.3f}]")
print(f"    差异 (合理: 不同初始化和权重导致)")

In [ ]:
# ============================================================
# Cell 7b: 中间特征维度检查
# ============================================================
# 验证: Backbone → GAP → Fusion → Head 的维度链条

sag = torch.randn(2, 5, 384, 384).to(device)
cor = torch.randn(2, 5, 384, 384).to(device)
ax = torch.randn(2, 5, 384, 384).to(device)

with torch.no_grad():
    # Step 1: backbone raw features
    raw_sag = model_shared.backbone.backbone.forward_features(sag)
    print(f"  Backbone.forward_features: {list(raw_sag.shape)}")
    
    # Step 2: GAP
    gap_sag = raw_sag.mean(dim=[2, 3])
    print(f"  After GAP:                {list(gap_sag.shape)}")
    
    # Step 3: extract_features
    f_sag = model_shared.backbone.extract_features(sag)
    print(f"  extract_features:         {list(f_sag.shape)}")
    assert torch.allclose(f_sag, gap_sag, atol=1e-5), "GAP mismatch in extract_features!"
    print(f"  OK: extract_features == GAP")

# Step 4: fusion
f_cor = model_shared.backbone.extract_features(cor)
f_ax = model_shared.backbone.extract_features(ax)
fused = model_shared.fusion(f_sag, f_cor, f_ax)
print(f"  After Fusion:             {list(fused.shape)}")

# Step 5: classification head
logits = model_shared.head(fused)
print(f"  After Head (logits):      {list(logits.shape)}")
print(f"  Logits value range:       [{logits.min():.3f}, {logits.max():.3f}]")
print(f"  After sigmoid range:      [{logits.sigmoid().min():.3f}, {logits.sigmoid().max():.3f}]")

In [ ]:
# ============================================================
# Cell 8: 缺失平面处理测试
# ============================================================
# 验证: 当某平面 DICOM 缺失时 (全零 tensor)，missing_emb 正确工作

sag_ok = torch.randn(4, 5, 384, 384).to(device)
cor_ok = torch.randn(4, 5, 384, 384).to(device)
ax_ok = torch.randn(4, 5, 384, 384).to(device)
zeros = torch.zeros(4, 5, 384, 384).to(device)

tests = [
    ("All planes OK",          sag_ok, cor_ok, ax_ok),
    ("Coronal MISSING",        sag_ok, zeros,  ax_ok),
    ("Axial MISSING",          sag_ok, cor_ok, zeros),
    ("Sag+Cor OK, Ax MISSING", sag_ok, cor_ok, zeros),
    ("All MISSING",            zeros,  zeros,  zeros),
]

print("缺失平面处理测试:")
for name, sag, cor, ax in tests:
    with torch.no_grad():
        out = model_shared(sag, cor, ax)
    print(f"  {name:<25s} → output shape={list(out.shape)}, range=[{out.min():.3f}, {out.max():.3f}]")

# 检查 missing_emb
print(f"\n  missing_emb (learnable): shape={list(model_shared.missing_emb.shape)}, norm={model_shared.missing_emb.norm().item():.4f}")
print(f"  OK: 缺失平面不导致崩溃，missing_emb 正确作为 fallback")

# 验证: 全零输入 vs 正常输入产生不同输出
with torch.no_grad():
    out_normal = model_shared(sag_ok, cor_ok, ax_ok)
    out_all_zero = model_shared(zeros, zeros, zeros)
diff = (out_normal - out_all_zero).abs().mean().item()
print(f"\n  正常 vs 全零输出差异 (MAE): {diff:.4f}")
print(f"  {'OK: 差异显著 (正常)' if diff > 0.01 else 'WARN: 差异太小，检查 missing_emb'}")

In [ ]:
# ============================================================
# Cell 8b: 混合 batch 测试 (部分 sample 缺平面)
# ============================================================
# 模拟: batch 中前 2 个 sample 缺 Coronal，后 2 个正常

sag_mixed = torch.randn(4, 5, 384, 384).to(device)
cor_mixed = torch.randn(4, 5, 384, 384).to(device)
ax_mixed = torch.randn(4, 5, 384, 384).to(device)

# 让 sample 1 和 3 的 Coronal 缺失
cor_mixed[1] = 0
cor_mixed[3] = 0

with torch.no_grad():
    out_mixed = model_shared(sag_mixed, cor_mixed, ax_mixed)
    
    # 提取平面特征看看
    feats = model_shared.get_plane_features(sag_mixed, cor_mixed, ax_mixed)
    
print("混合 batch 测试 (sample 1,3 缺 Coronal):")
for i in range(4):
    cor_norm = feats["cor"][i].norm().item()
    missing_mark = " ← MISSING (missing_emb)" if i in [1, 3] else ""
    print(f"  sample {i}: cor_feature_norm={cor_norm:.4f}{missing_mark}")

# 检查 missing samples 是否使用相同的 missing_emb
emb_sim = torch.cosine_similarity(feats["cor"][1:2], feats["cor"][3:4]).item()
print(f"\n  Missing sample 1 vs 3 cosine similarity: {emb_sim:.4f}")
print(f"  {'OK: 缺失样本使用相同的 missing_emb' if emb_sim > 0.99 else 'WARN: 缺失样本特征不一致'}")

In [ ]:
# ============================================================
# Cell 9: 融合模块对比 — Concat vs Transformer
# ============================================================
from models import MultiPlaneFusion

D = model_cfg["feature_dim"]

# 模拟三个平面的特征
f_sag_test = torch.randn(4, D).to(device)
f_cor_test = torch.randn(4, D).to(device)
f_ax_test = torch.randn(4, D).to(device)

# Concat fusion
fusion_concat = MultiPlaneFusion(feature_dim=D, fusion="concat").to(device)
out_concat = fusion_concat(f_sag_test, f_cor_test, f_ax_test)
print(f"Concat Fusion:")
print(f"  Input:  [{list(f_sag_test.shape)}] × 3")
print(f"  Concat: [B, {D*3}] → Linear → [B, {D}]")
print(f"  Output: {list(out_concat.shape)}")
print(f"  Params: {sum(p.numel() for p in fusion_concat.parameters()):,}")

# Transformer fusion
fusion_trans = MultiPlaneFusion(feature_dim=D, fusion="transformer", num_heads=4, num_layers=2).to(device)
out_trans = fusion_trans(f_sag_test, f_cor_test, f_ax_test)
print(f"\nTransformer Fusion:")
print(f"  Input:  [{list(f_sag_test.shape)}] × 3")
print(f"  3 tokens + position embed → TransformerEncoder(2 layers, 4 heads)")
print(f"  Output: {list(out_trans.shape)}")
print(f"  Params: {sum(p.numel() for p in fusion_trans.parameters()):,}")

# 对比输出差异
print(f"\n  两种融合的输出对比:")
print(f"    Concat:      range [{out_concat.min():.3f}, {out_concat.max():.3f}]")
print(f"    Transformer: range [{out_trans.min():.3f}, {out_trans.max():.3f}]")
print(f"    Cosine sim:  {torch.cosine_similarity(out_concat.mean(0, keepdim=True), out_trans.mean(0, keepdim=True)).item():.4f}")

del fusion_concat, fusion_trans

In [ ]:
# ============================================================
# Cell 9b: 各平面对融合输出的贡献分析
# ============================================================
# 验证: 每个平面是否对最终输出有独立贡献

with torch.no_grad():
    # Baseline: 三平面全 normal
    out_all = model_shared(sag_ok, cor_ok, ax_ok)
    
    # Ablation: 逐个去掉一个平面
    out_no_sag = model_shared(zeros, cor_ok, ax_ok)
    out_no_cor = model_shared(sag_ok, zeros, ax_ok)
    out_no_ax = model_shared(sag_ok, cor_ok, zeros)

print("平面 Ablation 分析 (移除某平面对 logits 的影响):")
for name, out_ab in [("No Sagittal", out_no_sag), ("No Coronal", out_no_cor), ("No Axial", out_no_ax)]:
    diff = (out_all - out_ab).abs().mean().item()
    print(f"  {name:<15s}  MAE vs full: {diff:.4f}")

print(f"\n  解释: MAE 越大 → 该平面对最终预测贡献越大")
print(f"  注意: 随机输入下差异来自模型权重差异，训练后会更显著")

In [ ]:
# ============================================================
# Cell 10: 过拟合测试 — Tri-Plane 单 batch 训练
# ============================================================
# 最重要的调试检查: 模型能否在少量样本上过拟合？
# 仅在 DICOM 数据可用时运行

if len(ds) > 0:
    from torch.utils.data import DataLoader
    from losses import FocalBCELoss
    from utils import compute_macro_auc

    print("Tri-Plane Overfitting Test")

    # 取一个小 batch
    loader = DataLoader(ds, batch_size=2, shuffle=True, num_workers=0)
    batch = next(iter(loader))
    sag = batch["sag"].to(device)
    cor = batch["cor"].to(device)
    ax = batch["ax"].to(device)
    lbls = batch["labels"].to(device)

    print(f"  Overfitting on {sag.shape[0]} tri-plane samples")

    # 新模型
    test_model = TriPlaneModel(
        in_channels=5, num_classes=12, pretrained=True,
        shared_backbone=True, fusion="concat",
    ).to(device)

    criterion = FocalBCELoss(gamma=2.0, alpha=0.25)
    optimizer = torch.optim.AdamW(test_model.parameters(), lr=1e-3)

    loss_history = []
    MAX_STEPS = 200

    for step in range(MAX_STEPS):
        test_model.train()
        optimizer.zero_grad()
        out = test_model(sag, cor, ax)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        loss_history.append(loss.item())

        if step % 40 == 0 or loss.item() < 0.001:
            with torch.no_grad():
                auc = compute_macro_auc(lbls.cpu().numpy(), out.detach().cpu().numpy())
            print(f"  step {step:4d}: loss={loss.item():.6f}  AUC={auc:.4f}")

        if loss.item() < 0.001:
            print(f"  Tri-plane 过拟合成功! (step {step})")
            break
    else:
        print(f"  {MAX_STEPS} steps 完成, final loss={loss_history[-1]:.6f}")
        if loss_history[-1] > 0.01:
            print(f"  WARN: 过拟合可能不充分，检查模型/数据/损失函数")

    # Loss 曲线
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(loss_history)
    ax.set_xlabel("Step"); ax.set_ylabel("Loss")
    ax.set_title("Tri-Plane Overfitting Test — Single Batch")
    ax.set_yscale("log"); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("⚠️  无 DICOM 数据，跳过过拟合测试。")
    print("   在 Kaggle 上运行时此 cell 将验证模型能否在单 batch 上收敛。")

In [ ]:
# ============================================================
# Cell 11: 快速干跑 — Tri-Plane 2 epochs (含梯度累积)
# ============================================================
# 最终验证: 完整 tri-plane 训练循环不报错 + 梯度累积正确

if len(ds) > 0:
    from torch.utils.data import DataLoader
    from datasets import TriPlaneDataset
    from train import train_one_epoch_triplane, validate_one_epoch_triplane
    from utils import format_per_class_auc

    # 构建小的 train 和 val datasets
    ds_tiny = TriPlaneDataset(series_df, train_df.head(8), **{
        "dicom_root": config["paths"]["dicom_root"],
        "image_size": config["data"]["image_size"],
        "slice_count": config["data"]["slice_count"],
        "is_train": True,
    })
    ds_val_tiny = TriPlaneDataset(series_df, val_df.head(4), **{
        "dicom_root": config["paths"]["dicom_root"],
        "image_size": config["data"]["image_size"],
        "slice_count": config["data"]["slice_count"],
        "is_train": False,
    })

    train_loader = DataLoader(ds_tiny, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(ds_val_tiny, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

    # 新模型
    dry_model = TriPlaneModel(
        in_channels=5, num_classes=12, pretrained=True,
        shared_backbone=True, fusion="concat",
    ).to(device)

    criterion = FocalBCELoss(gamma=2.0, alpha=0.25)
    optimizer = torch.optim.AdamW(dry_model.parameters(), lr=2e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None

    # 梯度累积步数
    accum_steps = config["train"]["gradient_accumulation_steps"]

    print(f"Tri-Plane 干跑: 2 epochs, train={len(ds_tiny)} val={len(ds_val_tiny)} samples")
    print(f"  batch_size=2, grad_accum={accum_steps}, effective_batch=2×{accum_steps}={2*accum_steps}")
    print(f"{'='*55}")

    for epoch in range(2):
        t0 = time.time()
        train_loss = train_one_epoch_triplane(
            dry_model, train_loader, optimizer, criterion, scaler,
            grad_clip_norm=1.0, grad_accum_steps=accum_steps, device=device,
        )
        val_metrics = validate_one_epoch_triplane(
            dry_model, val_loader, criterion, device=device,
        )

        elapsed = time.time() - t0
        vram = torch.cuda.max_memory_allocated(device) / 1024**3 if device == "cuda" else 0
        if device == "cuda":
            torch.cuda.reset_peak_memory_stats(device)

        print(f"Epoch {epoch}: train_loss={train_loss:.4f}  "
              f"val_loss={val_metrics['loss']:.4f}  val_auc={val_metrics['macro_auc']:.4f}  "
              f"time={elapsed:.0f}s  VRAM={vram:.1f}GB")
        print(f"         per-class: {format_per_class_auc(val_metrics['per_class_auc'])}")

    print(f"{'='*55}")
    print("Tri-Plane 2-epoch 干跑完成，训练循环无报错!")
else:
    print("⚠️  无 DICOM 数据，跳过训练干跑。")
    print("   在 Kaggle 上运行时此 cell 将验证 tri-plane 训练循环。")
    print("   本地可运行以下命令进行干跑:")
    print(f"   python train.py --config configs/triplane.yaml --epochs 2")

In [ ]:
# ============================================================
# Cell 12: 梯度累积验证
# ============================================================
# 验证: optimizer.step() 调用频率 = n_batches / grad_accum_steps
# 而非每个 batch 都 step

if len(ds) > 0:
    from torch.utils.data import DataLoader

    # 构建小 loader 用于验证
    test_accum_loader = DataLoader(ds_tiny, batch_size=2, shuffle=False, num_workers=0)

    # 钩子: 记录 optimizer.step() 调用次数
    step_count = [0]
    orig_step = optimizer.step
    def counting_step():
        step_count[0] += 1
        orig_step()
    optimizer.step = counting_step

    # 运行 1 个 epoch
    test_model_accum = TriPlaneModel(
        in_channels=5, num_classes=12, pretrained=True,
        shared_backbone=True, fusion="concat",
    ).to(device)

    criterion_test = FocalBCELoss(gamma=2.0, alpha=0.25)
    opt_test = torch.optim.AdamW(test_model_accum.parameters(), lr=1e-4)
    scaler_test = torch.amp.GradScaler("cuda") if device == "cuda" else None

    accum_steps = config["train"]["gradient_accumulation_steps"]
    n_batches = len(test_accum_loader)

    # 用钩子监控 step
    test_step_count = [0]
    test_orig_step = opt_test.step
    def test_counting_step():
        test_step_count[0] += 1
        test_orig_step()
    opt_test.step = test_counting_step

    train_loss = train_one_epoch_triplane(
        test_model_accum, test_accum_loader, opt_test, criterion_test, scaler_test,
        grad_clip_norm=1.0, grad_accum_steps=accum_steps, device=device,
    )

    expected_steps = (n_batches + accum_steps - 1) // accum_steps  # ceil division
    print(f"梯度累积验证:")
    print(f"  Batches:           {n_batches}")
    print(f"  Accum steps:       {accum_steps}")
    print(f"  Effective BS:      {2 * accum_steps}")
    print(f"  Optimizer steps:   {test_step_count[0]} (expected ~{expected_steps})")

    if test_step_count[0] <= expected_steps + 1:
        print(f"  OK: optimizer.step() 频率正确 (≈batches/accum_steps)")
    else:
        print(f"  WARN: optimizer.step() 调用次数过多, 梯度累积可能未生效")

    # 清理
    del test_model_accum, opt_test, criterion_test, scaler_test
else:
    print("⚠️  无 DICOM 数据，跳过梯度累积验证。")
    print("   原理: train_one_epoch_triplane 仅在每 accum_steps 个 batch 后调用 optimizer.step()")
    print(f"   配置: gradient_accumulation_steps={config['train']['gradient_accumulation_steps']}")
    print(f"   预期: optimizer.step() 频率 = 1/{config['train']['gradient_accumulation_steps']} 个 batch")

In [ ]:
---
## 调试结论

Cell 1-4 和 7-9 完全不需要 DICOM 即可测试。Cell 5-6, 10-14 需要 DICOM 或模型 checkpoint。

### 常见问题速查

| 现象 | 可能原因 | 检查 |
|------|----------|------|
| TriPlaneDataset 样本数为 0 | DICOM 路径不对或数据不存在 | Cell 4b 确认 study-DICOM 交集 |
| Cor/Ax 全零 | 该 study 缺少该平面 | Cell 8 验证 missing_emb 生效 |
| VRAM OOM | batch_size 太大 | tri-plane 用 bs=4 × accum=4 (VRAM friendly) |
| 三平面 AUC ≈ 单平面 | 融合权重退化 | Cell 9b ablation 分析 |
| Forward shape 错误 | _encode_plane 缺少 GAP | Cell 7b 逐步检查维度 |
| 训练不收敛 | LR 不合适或伪标签噪声 | Cell 10 过拟合测试 |
| DataLoader 慢 | volume cache 太小 | 增大 `_MAX_CACHE_SIZE` (triplane_dataset.py:26) |
| Loss 震荡 | 梯度累积未生效 | Cell 12 验证 optimizer.step 频率 |
| 推理结果全是 0.5 | 模型未正确加载或数据缺失 | Cell 14 检查 checkpoint 和 test DICOM |

### 运行完整训练

```bash
# Kaggle 上运行 (完整训练)
python train.py --config configs/triplane.yaml

# 快速验证 (2 epochs, 确认管线无报错)
python train.py --config configs/triplane.yaml --epochs 2

# 放宽置信度 (HIGH+MEDIUM → 更多训练样本, ~3800 studies)
python train.py --config configs/triplane.yaml --confidence HIGH_PLUS_MEDIUM

# 推理 (训练完成后)
python train.py --config configs/triplane.yaml \
    --inference outputs/triplane_efficientnetv2s/checkpoints/best_model.pt

# 推理 + 自定义 batch size + TTA
python train.py --config configs/triplane.yaml \
    --inference outputs/triplane_efficientnetv2s/checkpoints/best_model.pt \
    --batch_size 32
```

### Phase 2 文件结构

```
├── configs/triplane.yaml          ← 三平面训练配置 (batch=4, accum=4, shared bb)
├── models/triplane.py             ← TriPlaneModel (shared/indep backbone + missing_emb)
├── models/fusion.py               ← MultiPlaneFusion (concat / transformer)
├── models/efficientnet25d.py      ← EfficientNetV2S25D (5ch → extract_features)
├── models/head.py                 ← ClassificationHead
├── datasets/triplane_dataset.py   ← TriPlaneDataset (Sag anchor, LRU cache, 3-plane)
├── datasets/pseudo_labels.py      ← PseudoLabelLoader (合并 pseudo + gold)
├── losses/focal_bce.py            ← FocalBCELoss
├── train.py                       ← 训练/验证/推理 一体 (CLI 驱动)
└── notebooks/06_phase2_debug.ipynb ← 本调试 notebook
```

In [ ]:
# ============================================================
# Cell 12: 平面特征可视化 (PCA / t-SNE)
# ============================================================
# 仅当 DICOM 存在时有效 — 比较 Sag/Cor/Ax 特征空间

if len(ds) > 0:
    from sklearn.decomposition import PCA
    
    # 收集 10 个样本的各平面特征
    n_vis = min(10, len(ds))
    all_feats = {"sag": [], "cor": [], "ax": [], "fused": [], "labels": []}

    for i in range(n_vis):
        sample = ds[i * len(ds) // n_vis]
        sag = sample["sag"].unsqueeze(0).to(device)
        cor = sample["cor"].unsqueeze(0).to(device)
        ax = sample["ax"].unsqueeze(0).to(device)
        
        with torch.no_grad():
            feats = model_shared.get_plane_features(sag, cor, ax)
        
        for k in ["sag", "cor", "ax", "fused"]:
            all_feats[k].append(feats[k].cpu().numpy())
        all_feats["labels"].append(sample["labels"].numpy())

    # PCA 可视化
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    for ax_idx, (k1, k2, title) in enumerate([
        ("sag", "cor", "Sagittal vs Coronal"),
        ("sag", "ax", "Sagittal vs Axial"),
        ("cor", "ax", "Coronal vs Axial"),
    ]):
        ax = axes[ax_idx]
        feats_pair = np.concatenate([
            np.concatenate(all_feats[k1]),
            np.concatenate(all_feats[k2]),
        ])
        pca = PCA(n_components=2).fit_transform(feats_pair)
        n_each = len(all_feats[k1])
        ax.scatter(pca[:n_each, 0], pca[:n_each, 1], label=k1, alpha=0.7)
        ax.scatter(pca[n_each:, 0], pca[n_each:, 1], label=k2, alpha=0.7)
        ax.set_title(title); ax.legend()
    
    fig.suptitle("Plane Feature Space Comparison (PCA)", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  无 DICOM 数据，跳过特征可视化。")

---
## 调试结论

所有 12 项检查中，Cell 1-4 和 7-9 不需要 DICOM 即可完整测试。Cell 5-6 和 10-12 需要在 Kaggle 上运行。

### 常见问题速查

| 现象 | 可能原因 | 检查 |
|------|----------|------|
| TriPlaneDataset 样本数为 0 | DICOM 路径不对或数据不存在 | Cell 4b 确认 study-DICOM 交集 |
| Cor/Ax 全零 | 该 study 缺少该平面 | Cell 8 验证 missing_emb 生效 |
| VRAM OOM | batch_size 太大 | tri-plane 用 bs=4, 单平面用 bs=8 |
| 三平面 AUC ≈ 单平面 | 融合权重退化 | Cell 9b ablation 分析 |
| Forward shape 错误 | _encode_plane 缺少 GAP | Cell 7b 逐步检查维度 |
| 训练不收敛 | LR 不合适或伪标签噪声 | Cell 10 过拟合测试 |
| DataLoader 慢 | volume cache 太小 | 增大 `_MAX_CACHE_SIZE` |

### 运行完整训练

```bash
# Kaggle 上运行
python train.py --config configs/triplane.yaml

# 快速验证 (2 epochs)
python train.py --config configs/triplane.yaml --epochs 2

# 放宽置信度 (更多训练样本)
python train.py --config configs/triplane.yaml --confidence HIGH_PLUS_MEDIUM
```